# Explore: 익숙하지 않은 codebase에서 근거 확인하기

이 notebook은 agent를 한 번도 본 적 없는 repository에 투입하고 실제 architecture를 파악하도록 요청합니다. filesystem은 agent의 유일한 작업 공간이며, 선택하여 읽은 파일만 context window에 들어가므로 `ls`, `grep`, `read`를 사용한 탐색으로 머릿속 구조를 만들어 갑니다.

흥미로운 부분은 fixture에 심어 둔 함정입니다. `ARCHITECTURE.md`에는 코드가 더 이상 따르지 않는 구조가 설명되어 있어, 코드를 확인하지 않고 문서를 믿는 agent는 확신에 차서 틀린 답을 하게 됩니다. 이 맥락에서 grounding은 문서를 권위 있는 자료로 취급하는 대신 읽은 내용을 실제 존재하는 것과 대조하여 검증하는 것을 뜻합니다.

iterate notebook에 더하여 배울 내용은 다음과 같습니다.

- **행동 전에 탐색하기.** 좋은 agent는 tree를 충분히 읽고 이해한 다음 답하며, 그 반대 순서로 하지 않습니다.
- **세션 도중 resource 추가하기.** 마지막의 보충 설명에서는 세션을 다시 생성하는 대신 `sessions.resources.add`로 실행 중인 세션에 파일을 추가하는 방법을 보여 줍니다. 탐색을 통해 agent가 다음에 살펴봐야 할 무언가를 발견했을 때 유용합니다.


In [ ]:
import io
import os

from anthropic import Anthropic
from utilities import (
    make_unfamiliar_repo_zip,
    stream_until_end_turn,
    wait_for_idle_status,
)

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")

client = Anthropic()

## 1. repo fixture 생성하기

repo가 충분히 작으므로 notebook 옆에 디스크 fixture를 보관하는 대신 helper로 메모리에 만듭니다. helper는 `services/` microservices 구조와 아직 이전 monolithic 구조를 설명하는 오래된 `ARCHITECTURE.md`를 심습니다.


In [ ]:
buf = make_unfamiliar_repo_zip()
fixture_zip = client.beta.files.upload(file=("repo.zip", buf, "application/zip"))
print(f"fixture: {fixture_zip.id}")

## 2. Agent + environment + session


In [ ]:
agent = client.beta.agents.create(
    name="cookbook-explore",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-explore-unfamiliar-codebase"},
    model=MODEL,
    system=(
        "You are onboarding to an unfamiliar codebase. Explore before "
        "answering, docs can be stale. Verify what you read against "
        "actual code structure. Write notes to /tmp/NOTES.md as you go."
    ),
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
        }
    ],
)

env = client.beta.environments.create(
    name="cookbook-explore-env",
    config={"type": "cloud", "networking": {"type": "limited"}},
)

session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=[{"type": "file", "file_id": fixture_zip.id, "mount_path": "repo.zip"}],
    title="Onboard to repo",
)
print(f"session: {session.id}")

## 3. 탐색하고 오래된 문서의 함정 살펴보기

근거를 확인한 답은 실제 `services/` 구조를 언급하고 `ARCHITECTURE.md`가 오래되었다고 지적합니다. 근거를 확인하지 않은 답은 오래된 문서가 설명하는 monolith 구조를 그대로 되풀이합니다.


In [ ]:
client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Unzip /mnt/session/uploads/repo.zip to /tmp/repo/. "
                        "Then: what is the actual architecture of this "
                        "codebase? Be specific about directory structure. "
                        "Check if the docs are accurate."
                    ),
                }
            ],
        }
    ],
)

print("=== exploration ===")
stream_until_end_turn(client, session.id)

## 4. agent의 메모 다시 읽기

agent는 작업하면서 `/tmp/NOTES.md`에 메모를 남기도록 지시받았습니다. 이 파일을 출력하면 탐색 과정에서 codebase에 대한 이해가 어떻게 발전했는지 확인하는 데 유용합니다.


In [ ]:
client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [{"type": "text", "text": "cat /tmp/NOTES.md"}],
        }
    ],
)
stream_until_end_turn(client, session.id)

## 보충 설명: 실행 중인 세션에 context 추가하기

`sessions.create`의 `resources=` 인자는 파일을 mount하는 가장 일반적인 방법이지만, API는 기존 세션의 mount를 관리하는 `/v1/sessions/<id>/resources` sub-resource도 제공합니다. 여기서 유용합니다. 탐색 중 추가 context(config 파일, changelog, 외부 schema)가 필요한 질문이 드러나면 세션을 종료하지 않고 추가할 수 있습니다.

패턴은 이미 알고 있는 업로드 후 연결하는 루프와 같으며, 호출 하나 대신 두 개로 나뉘어 있을 뿐입니다.


In [ ]:
hints = b"# DEPLOY HISTORY\n2026-03-01: monolith -> microservices migration complete\n"
hints_file = client.beta.files.upload(
    file=("DEPLOY_HISTORY.md", io.BytesIO(hints), "text/markdown")
)

added = client.beta.sessions.resources.add(
    session_id=session.id,
    type="file",
    file_id=hints_file.id,
    mount_path="DEPLOY_HISTORY.md",
)
print(f"added resource {added.id} to session {session.id}")

attached = client.beta.sessions.resources.list(session_id=session.id)
print(f"{len(attached.data)} resources attached now")

client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "There's a DEPLOY_HISTORY.md in your workspace now. "
                        "Read it and tell me whether it changes anything in "
                        "your earlier answer."
                    ),
                }
            ],
        }
    ],
)
print("\n--- follow-up with deploy history ---")
stream_until_end_turn(client, session.id)

# agent가 사용을 마쳤으므로 이제 파일 연결을 해제합니다. 여기의 `delete`는
# resource 연결 해제 동작이며 cookbook 전체의 archive가 아닙니다.
client.beta.sessions.resources.delete(session_id=session.id, resource_id=added.id)
print("detached follow-up resource")

## 정리하기


In [ ]:
wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
print("archived")